**Student:** Allan Antonio Canales Corea

# Lesson 6 Mini-Project: Open Data ELT Workflow (part 2)
### Daily weather in Lisbon from the Open-Meteo API, cleaned with pandas

**Goal:** build a small data workflow with an open API and save a clean CSV for a later dashboard.

**Workflow (ELT):** **Extract (API) -> Load raw CSV -> Transform with pandas -> Load processed CSV**

**Data source:** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). It is free and needs no API key.

I chose Lisbon because I live in Portugal, so the numbers are easy for me to understand.

## 1 - Imports and settings
We use `requests` for the API and `pandas` for the tables. The settings are in one place so they are easy to change.

In [1]:
import os
import requests
import pandas as pd

LATITUDE, LONGITUDE = 38.72, -9.14      # Lisbon
START_DATE, END_DATE = "2025-01-01", "2025-12-31"
RAW_PATH = "data/raw/raw_weather_lisbon.csv"
OUT_PATH = "data/processed/output.csv"
SUMMARY_PATH = "data/processed/monthly_summary.csv"

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
print("Ready")

Ready


## 2 - Extract: call the API
We ask for daily maximum and minimum temperature, total rain and maximum wind for the whole year 2025. We use `raise_for_status()` as basic error handling and check that the response has data.

In [2]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
    "timezone": "Europe/Lisbon",
}

try:
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as error:
    raise SystemExit(f"API request failed: {error}")

print("Status code:", response.status_code)
print("Keys in the JSON:", list(data.keys()))
print("Units:", data["daily_units"])

Status code: 200
Keys in the JSON: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Units: {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}


The JSON has a `daily` part where every variable is a list with one value per day. This shape can be turned into a DataFrame directly.

In [3]:
df_raw = pd.DataFrame(data["daily"])
print("Shape (rows, columns):", df_raw.shape)
display(df_raw.head())

Shape (rows, columns): (365, 5)


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
0,2025-01-01,12.9,6.6,0.0,15.4
1,2025-01-02,15.0,5.8,0.0,10.0
2,2025-01-03,16.1,8.6,1.7,12.4
3,2025-01-04,17.4,10.8,1.3,21.4
4,2025-01-05,16.1,13.4,25.3,33.2


## 3 - Load (raw)
In ELT we first save the data exactly as it came, before changing anything. This way we can always go back to the original.

In [4]:
df_raw.to_csv(RAW_PATH, index=False)
print("Saved raw file:", RAW_PATH, "with", len(df_raw), "rows")

Saved raw file: data/raw/raw_weather_lisbon.csv with 365 rows


## 4 - Data checks
Before transforming, we check the quality: types, missing values and duplicated days.

In [5]:
print(df_raw.dtypes)
print()
print("Missing values per column:")
print(df_raw.isna().sum())
print()
print("Duplicated dates:", df_raw["time"].duplicated().sum())
display(df_raw.describe().round(2))

time                      str
temperature_2m_max    float64
temperature_2m_min    float64
precipitation_sum     float64
wind_speed_10m_max    float64
dtype: object

Missing values per column:
time                  0
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
wind_speed_10m_max    0
dtype: int64

Duplicated dates: 0


,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
count,365.00,365.00,365.00,365.00
mean,22.12,14.07,2.67,18.07
std,6.61,4.06,6.64,6.08
min,11.40,5.00,0.00,6.60
25%,16.40,10.80,0.00,14.10
50%,20.40,13.90,0.00,17.00
75%,27.20,17.60,1.50,20.90
max,40.70,25.40,54.60,48.80


**Check result:** the API returned 365 rows (one per day of 2025) and 5 columns. There are 0 missing values and 0 duplicated dates. The `time` column is text, so it must be converted to a real date in the transformation step.

## 5 - Transform
We apply several transformations:
1. **Rename** the columns to short, clear names.
2. **Convert** `date` to a real datetime type.
3. **Compute new columns:** average temperature, temperature range, month, month name, and a `rainy_day` flag.
4. **Filter** out rows with missing values (if any).
5. **Aggregate** by month for a second table.

In [6]:
df = df_raw.copy()

# 1. Rename columns
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max_c",
    "temperature_2m_min": "temp_min_c",
    "precipitation_sum": "rain_mm",
    "wind_speed_10m_max": "wind_max_kmh",
})

# 2. Convert date text to datetime
df["date"] = pd.to_datetime(df["date"])

# 3. New columns
df["temp_avg_c"] = ((df["temp_max_c"] + df["temp_min_c"]) / 2).round(2)
df["temp_range_c"] = (df["temp_max_c"] - df["temp_min_c"]).round(2)
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%B")
df["rainy_day"] = df["rain_mm"] >= 1     # 1 mm or more counts as a rainy day

# 4. Remove rows with missing values
rows_before = len(df)
df = df.dropna().reset_index(drop=True)
print("Rows before:", rows_before, "| rows after dropna:", len(df))

display(df.head())

Rows before: 365 | rows after dropna: 365


,date,temp_max_c,temp_min_c,rain_mm,wind_max_kmh,temp_avg_c,temp_range_c,month,month_name,rainy_day
0,2025-01-01,12.9,6.6,0.0,15.4,9.75,6.3,1,January,False
1,2025-01-02,15.0,5.8,0.0,10.0,10.40,9.2,1,January,False
2,2025-01-03,16.1,8.6,1.7,12.4,12.35,7.5,1,January,True
3,2025-01-04,17.4,10.8,1.3,21.4,14.10,6.6,1,January,True
4,2025-01-05,16.1,13.4,25.3,33.2,14.75,2.7,1,January,True


Each step is a simple change. The new columns make the data more useful for a dashboard, for example to compare months or count rainy days.

In [7]:
# 5. Aggregate by month
monthly = df.groupby(["month", "month_name"]).agg(
    days=("date", "count"),
    avg_temp_c=("temp_avg_c", "mean"),
    max_temp_c=("temp_max_c", "max"),
    min_temp_c=("temp_min_c", "min"),
    total_rain_mm=("rain_mm", "sum"),
    rainy_days=("rainy_day", "sum"),
).round(2).reset_index()

display(monthly)

,month,month_name,days,avg_temp_c,max_temp_c,min_temp_c,total_rain_mm,rainy_days
0,1,January,31,13.18,19.6,5.0,160.7,14
1,2,February,28,13.00,18.5,7.2,92.1,9
2,3,March,31,13.37,23.6,7.9,168.3,20
3,4,April,30,16.11,26.4,9.6,116.2,19
4,5,May,31,18.90,35.5,11.0,40.3,4
5,6,June,30,23.21,40.7,15.0,2.5,0
6,7,July,31,24.33,37.4,16.7,2.8,1
7,8,August,31,24.79,38.3,17.3,0.4,0
8,9,September,30,21.88,33.9,13.2,14.9,2
9,10,October,31,20.92,29.8,14.0,48.6,8


## 6 - Load (processed)
We save the clean daily table as `data/processed/output.csv` (the file the homework asks for) and the monthly summary as an extra file.

In [8]:
df.to_csv(OUT_PATH, index=False)
monthly.to_csv(SUMMARY_PATH, index=False)
print("Saved:", OUT_PATH, df.shape)
print("Saved:", SUMMARY_PATH, monthly.shape)

# Read the file back to confirm it works
check = pd.read_csv(OUT_PATH)
print("Rows read back:", len(check))
display(check.head())

Saved: data/processed/output.csv (365, 10)
Saved: data/processed/monthly_summary.csv (12, 8)
Rows read back: 365


,date,temp_max_c,temp_min_c,rain_mm,wind_max_kmh,temp_avg_c,temp_range_c,month,month_name,rainy_day
0,2025-01-01,12.9,6.6,0.0,15.4,9.75,6.3,1,January,False
1,2025-01-02,15.0,5.8,0.0,10.0,10.40,9.2,1,January,False
2,2025-01-03,16.1,8.6,1.7,12.4,12.35,7.5,1,January,True
3,2025-01-04,17.4,10.8,1.3,21.4,14.10,6.6,1,January,True
4,2025-01-05,16.1,13.4,25.3,33.2,14.75,2.7,1,January,True


## 7 - Results and conclusions

In [9]:
hottest = df.loc[df["temp_max_c"].idxmax()]
coldest = df.loc[df["temp_min_c"].idxmin()]
wettest = df.loc[df["rain_mm"].idxmax()]
windiest = df.loc[df["wind_max_kmh"].idxmax()]

print(f"Days in the dataset: {len(df)}")
print(f"Hottest day: {hottest['date'].date()} with {hottest['temp_max_c']} C")
print(f"Coldest night: {coldest['date'].date()} with {coldest['temp_min_c']} C")
print(f"Wettest day: {wettest['date'].date()} with {wettest['rain_mm']} mm")
print(f"Windiest day: {windiest['date'].date()} with {windiest['wind_max_kmh']} km/h")
print(f"Average temperature of the year: {df['temp_avg_c'].mean():.2f} C")
print(f"Total rain: {df['rain_mm'].sum():.1f} mm in {int(df['rainy_day'].sum())} rainy days")
print("Warmest month:", monthly.loc[monthly['avg_temp_c'].idxmax(), 'month_name'])
print("Coolest month:", monthly.loc[monthly['avg_temp_c'].idxmin(), 'month_name'])
print("Rainiest month:", monthly.loc[monthly['total_rain_mm'].idxmax(), 'month_name'])

Days in the dataset: 365
Hottest day: 2025-06-29 with 40.7 C
Coldest night: 2025-01-15 with 5.0 C
Wettest day: 2025-11-13 with 54.6 mm
Windiest day: 2025-03-20 with 48.8 km/h
Average temperature of the year: 18.09 C
Total rain: 975.1 mm in 107 rainy days
Warmest month: August
Coolest month: December
Rainiest month: November


**Conclusions (based on my own results):**
- The workflow worked end to end: 365 daily rows came from the API, were saved raw, cleaned and saved as `data/processed/output.csv` (365 rows, 10 columns), plus a 12-row monthly summary.
- Lisbon in 2025 had an average temperature of 18.09 C. The hottest day was 2025-06-29 with 40.7 C and the coldest night was 2025-01-15 with 5.0 C.
- August was the warmest month (24.79 C on average) and December the coolest (12.28 C).
- It rained 975.1 mm in total, on 107 rainy days (1 mm or more). November was the rainiest month with 193.6 mm, and the wettest single day was 2025-11-13 with 54.6 mm. June and August had no rainy days at all.
- Summer is very dry and winter is wet, which is the typical Mediterranean pattern. A dashboard could use `output.csv` for daily lines and `monthly_summary.csv` for monthly bars.
- Limitation: this is only one city and one year, so it shows 2025 and not a long-term climate.